In [1]:
from bs4 import BeautifulSoup
import chardet
import datetime
import duckdb
import io
from openpyxl import load_workbook
from openpyxl import load_workbook
import pandas as pd
import requests
import xlrd

In [2]:
DATA_PATH = '../data/'
PATH_TO_DB = DATA_PATH+'creatives.duckdb'

PATH_TO_XCL = DATA_PATH+'excel/arts_council/'                # input
FILENAMES = [
    "Grant_commitments_2022-23_Qtr1.xlsx",
    "Grant_commitments_2022-23_Qtr3_YTD.xlsx",
    "Grant_commitments_2022-23_Qtr4_YTD_0.xlsx",
    "Grant_commitments_2022-23_YTD_Qtr2.xlsx",
    "Grant_commitments_2023-24_Q2_0.xlsx",
    "Grant_commitments_2023-24_Qtr3_YTD.xlsx",
    "Grant_commitments_2024-25_Qtr2_YTD.xlsx",
    "Grant_commitments_2024-25_Qtr3_YTD.xlsx"
]

# assumptions: 
#    - 6 decorative rows
#    - row 7 is header
#    - that the column count < 26^2
HEADER_ROW=7
HEADER_CELL_RANGE_START='A7'
HEADER_CELL_RANGE_END='ZZ7'




In [3]:
filename=FILENAMES[0]
tablename=filename.split('.')[0].lower()
tablename

'grant_commitments_2022-23_qtr1'

In [4]:
%%time
for filename in FILENAMES:
    filepath = PATH_TO_XCL + filename
    try:
        wb =load_workbook(filename=filepath, data_only=True) 
        print('excel workbook at', filepath, 'contains the following sheet(s)', wb.sheetnames)
        sheet = wb.active
        print('determine the header row for sheet', sheet.title)
        try:
            cell_range = sheet[HEADER_CELL_RANGE_START:HEADER_CELL_RANGE_END] 
            header_row = [[cell.value for cell in component if cell.value] for component in cell_range][0]
            num_columns = len(header_row)
            print(num_columns, 'columns:', header_row)
            # set up a data frame to collect table values into
            df = pd.DataFrame(columns=header_row) 
            try: 
                # append table, one row at a time. skip the first 7 rows (6 rows of decoration and the header row) 
                for row in sheet.iter_rows(min_row=8):
                    data_row = [element.value for element in row]
                    if data_row.count(None) == len(data_row): # if all cells in row are NULL
                        continue
                    else: # at least one element with non-NULL value
                        df.loc[len(df)] = data_row[:num_columns]
                # dataframe populated
                try: # export to database
                    tablename=filename.split('.')[0].lower()
                    with duckdb.connect(database=PATH_TO_DB, read_only=False) as con:
                        con.sql(f"DROP TABLE IF EXISTS {tablename};")
                        con.sql(f"CREATE TABLE {tablename} AS SELECT * FROM df;")
                        con.sql(f"SELECT COUNT(*) FROM {tablename};")
                except:
                    print('error loading df to database')
            except:
                print('error reading data rows')
        except:
            print('error reading header')
    except:
        print('error loading workbook')

/Users/oh/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/Users/oh/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/openpyxl/reader/drawings.py:67: UserWarning: wmf image format is not supported so the image is being dropped
  warn(msg)


excel workbook at ../data/excel/arts_council/Grant_commitments_2022-23_Qtr1.xlsx contains the following sheet(s) ['2023 Qtr1 ', 'Summary']
determine the header row for sheet 2023 Qtr1 
11 columns: ['Budget Year', 'Applicant Name', 'Funding Source', 'Programme Name', 'Budget Heading', 'Grant Amount', 'Main Artform', 'Local Authority', 'Region', 'Constituency', ' Ward']
error loading df to database


/Users/oh/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/Users/oh/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/openpyxl/reader/drawings.py:67: UserWarning: wmf image format is not supported so the image is being dropped
  warn(msg)


excel workbook at ../data/excel/arts_council/Grant_commitments_2022-23_Qtr3_YTD.xlsx contains the following sheet(s) ['2023 Qtr3 ', 'Summary']
determine the header row for sheet 2023 Qtr3 
0 columns: []
error reading data rows


/Users/oh/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/Users/oh/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/openpyxl/reader/drawings.py:67: UserWarning: wmf image format is not supported so the image is being dropped
  warn(msg)


...

excel workbook at ../data/excel/arts_council/Grant_commitments_2023-24_Qtr3_YTD.xlsx contains the following sheet(s) ['2024 Qtr3', 'Summary']
determine the header row for sheet 2024 Qtr3
0 columns: []
error reading data rows


/Users/oh/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/Users/oh/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/openpyxl/reader/drawings.py:67: UserWarning: wmf image format is not supported so the image is being dropped
  warn(msg)


excel workbook at ../data/excel/arts_council/Grant_commitments_2024-25_Qtr2_YTD.xlsx contains the following sheet(s) ['24.25 Qtr2 YTD ', 'Summary']
determine the header row for sheet 24.25 Qtr2 YTD 
0 columns: []
error reading data rows


/Users/oh/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
/Users/oh/Library/jupyterlab-desktop/jlab_server/lib/python3.8/site-packages/openpyxl/reader/drawings.py:67: UserWarning: wmf image format is not supported so the image is being dropped
  warn(msg)


excel workbook at ../data/excel/arts_council/Grant_commitments_2024-25_Qtr3_YTD.xlsx contains the following sheet(s) ['24.25 Qtr3 YTD ', 'Summary']
determine the header row for sheet 24.25 Qtr3 YTD 
0 columns: []
error reading data rows
CPU times: user 1min 41s, sys: 2.24 s, total: 1min 43s
Wall time: 1min 44s


In [5]:
df.sample()

ValueError: a must be greater than 0 unless no samples are taken

In [ ]:
df.tail()